In [300]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

Importing Dataset:

In [301]:
df = pd.read_csv("btc_daily_data.csv")

df['Date'] = pd.to_datetime(df['Date'])

Feature engineering

In [302]:
df["day_of_week"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month
df["day_of_month"] = df["Date"].dt.day
df["close_lag1"] = df["Close"].shift(1)
df["close_lag2"] = df["Close"].shift(2)
df["volume_lag1"] = df["Volume"].shift(1)

df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)




df['return'] = df['Close'].pct_change()

df["price_range"] = df["High"] - df["Low"]
df["body_size"] = df["Close"] - df["Open"]

df['ma7'] = df['Close'].rolling(7).mean()
df['ma14'] = df['Close'].rolling(14).mean()

# df["trend"] = df["ma7"] - df["ma21"]

# df['volatility'] = df['return'].rolling(10).std()

# df['range'] = (df['High'] - df['Low']) / df['Close']

# df['volume_change'] = df['Volume'].pct_change()

# df['future_return'] = df['Close'].shift(-1) / df['Close'] - 1
# df['target'] = 0
# df.loc[df['future_return'] > 0.005, 'target'] = 1
# df.loc[df['future_return'] < -0.005, 'target'] = -1
# df = df[df['target'] != 0]   # remove small noise days


# delta = df["Close"].diff()
# gain = delta.clip(lower=0)
# loss = -delta.clip(upper=0)
# avg_gain = gain.rolling(14).mean()
# avg_loss = loss.rolling(14).mean()
# rs = avg_gain / avg_loss
# df["RSI"] = 100 - (100 / (1 + rs))

df['momentum_7'] = df['Close'] - df['Close'].shift(10)

# df['ma_ratio'] = df['ma7'] / df['ma14']

# df['volatility_14'] = df['return'].rolling(14).std()

# df['range_strength'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])


In [303]:
# features = [
#     'return',
#     'ma7',
#     'ma14',
#     'volatility',
#     'range',
#     'volume_change',
#     'rsi',
#     'momentum_7',
#     'ma_ratio',
#     'volatility_14',
#     'range_strength'
# ]

df = df.dropna()

X = df.drop(columns=['target','Date','Open','High'])
y = df['target']

In [304]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()
y = y.loc[X.index]

In [305]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))
print(y.value_counts(normalize=True))

Accuracy: 0.5169409486931268
target
1    0.531462
0    0.468538
Name: proportion, dtype: float64


In [306]:
baseline = y_test.value_counts(normalize=True).max()
print("Baseline:", baseline)

Baseline: 0.510164569215876


In [307]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, model.predict(X_test)))

[[417  89]
 [410 117]]


In [308]:
import json

model_data = {
    "weights": model.coef_[0].tolist(),
    "bias": model.intercept_[0],
    "features": model.features
}

with open("model.json", "w") as f:
    json.dump(model_data, f)

AttributeError: 'LogisticRegression' object has no attribute 'features'